In [15]:
# ─────────────────────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────────────────────
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# ─────────────────────────────────────────────────────────────
# 2. LOAD DATASET
# ─────────────────────────────────────────────────────────────
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

DATASET_PATH = "/kaggle/input/datasets/afianasumudeen/dermascan-acne-dataset/data/dermacon_acne"

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Classes:", train_ds.class_names)

# ─────────────────────────────────────────────────────────────
# 3. AUGMENTATION + PREPROCESSING
# ─────────────────────────────────────────────────────────────
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.map(lambda x, y: (preprocess(x), y))
val_ds   = val_ds.map(lambda x, y: (preprocess(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)

print("✅ Dataset ready")

# ─────────────────────────────────────────────────────────────
# 4. BUILD MODEL
# ─────────────────────────────────────────────────────────────
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.6),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(3, activation='softmax')
])

print("✅ Model built")

# ─────────────────────────────────────────────────────────────
# 5. CLASS WEIGHTS
# ─────────────────────────────────────────────────────────────
y = np.concatenate([y.numpy() for _, y in train_ds])

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
class_weights = dict(enumerate(class_weights))

# ─────────────────────────────────────────────────────────────
# 6. COMPILE (PHASE 1)
# ─────────────────────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 👉 BEFORE TRAINING ACCURACY
loss_before, acc_before = model.evaluate(val_ds, verbose=0)
print(f"\n📊 Before Training Accuracy: {acc_before:.4f}")

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

# ─────────────────────────────────────────────────────────────
# 7. PHASE 1 TRAINING
# ─────────────────────────────────────────────────────────────
print("\n🔥 Phase 1: Training head...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks
)

# ─────────────────────────────────────────────────────────────
# 8. FINE-TUNING
# ─────────────────────────────────────────────────────────────
base_model.trainable = True

FREEZE_UNTIL = 130

for layer in base_model.layers[:FREEZE_UNTIL]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n🔥 Phase 2: Fine-tuning...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks
)

# ─────────────────────────────────────────────────────────────
# 9. FINAL EVALUATION
# ─────────────────────────────────────────────────────────────
loss_after, acc_after = model.evaluate(val_ds, verbose=0)

print(f"\n📊 Before Training: {acc_before:.4f}")
print(f"📊 After Training : {acc_after:.4f}")
print(f"📈 Improvement    : {(acc_after - acc_before)*100:.2f}%")

Found 226 files belonging to 3 classes.
Using 181 files for training.
Found 226 files belonging to 3 classes.
Using 45 files for validation.
Classes: ['mild', 'moderate', 'severe']
✅ Dataset ready
✅ Model built

📊 Before Training Accuracy: 0.2667

🔥 Phase 1: Training head...
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - accuracy: 0.3276 - loss: 1.9009 - val_accuracy: 0.2667 - val_loss: 1.4781 - learning_rate: 0.0010
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 988ms/step - accuracy: 0.3183 - loss: 2.4385 - val_accuracy: 0.2444 - val_loss: 1.3673 - learning_rate: 0.0010
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.4498 - loss: 1.4327 - val_accuracy: 0.2444 - val_loss: 1.4155 - learning_rate: 0.0010
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.4405 - loss: 1.1882 - val_accuracy: 0.2889 - val_loss: 1.4696 - learning_rate: 0.0010
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 971ms/step - accuracy: 0.4503 - loss: 2.0439 - val_accuracy: 0.3111 - val_loss: 1.4528 - le

In [16]:
model.save("acne_model_final.keras")
print("✅ Model saved as acne_model_final.keras")

✅ Model saved as acne_model_final.keras


In [17]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optional optimizations (recommended)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# Save TFLite file
with open("acne_model.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ TFLite model saved as acne_model.tflite")

INFO:tensorflow:Assets written to: /tmp/tmpvlpap5_z/assets


INFO:tensorflow:Assets written to: /tmp/tmpvlpap5_z/assets


Saved artifact at '/tmp/tmpvlpap5_z'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_1922')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  134951474088848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951486996112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951486995344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951474089616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951486994192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951486993232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951486992848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951475323024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951486995728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951486995920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134951475

W0000 00:00:1774381254.941994      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774381254.942075      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1774381255.136051      55 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


In [4]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    for file in files:
        print("   ", file)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/afianasumudeen
/kaggle/input/datasets/afianasumudeen/model-acne
    copy_of_acne_model_best.keras
/kaggle/input/datasets/afianasumudeen/dermascan-acne-dataset
    acne_model_best.keras
/kaggle/input/datasets/afianasumudeen/dermascan-acne-dataset/data
/kaggle/input/datasets/afianasumudeen/dermascan-acne-dataset/data/dermacon_acne
/kaggle/input/datasets/afianasumudeen/dermascan-acne-dataset/data/dermacon_acne/mild
    IMG_3542_1.jpg
    IMG_3030.jpg
    IMG_3343_1.jpg
    IMG_0011.jpg
    IMG_6238_1.jpg
    IMG_5886_1.jpg
    IMG_2986_1.jpg
    IMG_0682.jpg
    IMG_5240.jpg
    IMG_1355.jpg
    IMG_3689_1.jpg
    IMG_1354.jpg
    IMG_1793.jpg
    IMG_6161_2.jpg
    IMG_1143.jpg
    IMG_1221.jpg
    IMG_5990_1.jpg
    IMG_3141.jpg
    IMG_4937.jpg
    IMG_0158.jpg
    IMG_1850.jpg
    IMG_5625_1.jpg
    IMG_2427.jpg
    IMG_0948.jpg
    IMG_4306.jpg
    IMG_3931_1.jpg
    IMG_0899.jpg
    IMG_5241_2.jpg
    IMG_3345.jpg
    IMG_5